In [ ]:

from pathlib import Path
import re
import numpy as np
import pandas as pd

# ---------- File paths ----------
try:
    NOTEBOOK_PATH = Path(__file__).resolve()
    ROOT = NOTEBOOK_PATH.parent.parent
except NameError:
    ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data"
REPORT_DIR = ROOT / "reports"

DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Use the provided dataset path.
INPUT_FILE = Path(r"C:\Users\bhush\Downloads\archive (3)\DataCoSupplyChainDataset.csv")

OUTPUT_FILE = DATA_DIR / "cleaned_supply_chain.csv"

# ---------- Load data ----------
if not INPUT_FILE.exists():
    available_csvs = sorted(
        str(path.relative_to(ROOT)) for path in ROOT.rglob("*.csv")
    )
    message = [
        f"Dataset not found: {INPUT_FILE}",
        "Check that the file path is correct and that the CSV is accessible from this machine.",
    ]
    if available_csvs:
        message.append("CSV files currently found in the workspace:")
        message.extend(f"- {path}" for path in available_csvs)
    else:
        message.append("No CSV files were found anywhere in the workspace.")
    raise FileNotFoundError("\n".join(message))

df = pd.read_csv(INPUT_FILE, encoding="latin1", low_memory=False)

print("Original dataset shape:", df.shape)

# ---------- 1. Standardize column names ----------
def standardize_column(name):
    name = str(name).strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")

df.columns = [standardize_column(c) for c in df.columns]

# Detect columns that became identical after standardization.
if df.columns.duplicated().any():
    duplicates = df.columns[df.columns.duplicated()].tolist()
    raise ValueError(
        f"Duplicate column names after standardization: {duplicates}"
    )

print("\nStandardized columns:")
print(df.columns.tolist())

# ---------- 2. Convert date columns ----------
date_columns = [
    c for c in df.columns
    if "date" in c or c.endswith("_datetime")
]

date_report = []

for col in date_columns:
    original_non_null = df[col].notna().sum()

    # Mixed date formats may occur in CSV datasets.
    try:
        converted = pd.to_datetime(
            df[col], errors="coerce", format="mixed"
        )
    except (TypeError, ValueError):
        converted = pd.to_datetime(df[col], errors="coerce")

    failed = int(
        (df[col].notna() & converted.isna()).sum()
    )

    df[col] = converted

    date_report.append({
        "column": col,
        "original_non_null": int(original_non_null),
        "failed_date_conversions": failed
    })

pd.DataFrame(date_report).to_csv(
    REPORT_DIR / "date_conversion_report.csv", index=False
)

# ---------- 3. Check duplicates and missing values ----------
exact_duplicate_count = int(df.duplicated().sum())

missing_report = pd.DataFrame({
    "column": df.columns,
    "missing_count": df.isna().sum().values,
    "missing_percentage": (
        df.isna().mean().values * 100
    ).round(2)
}).sort_values("missing_count", ascending=False)

missing_report.to_csv(
    REPORT_DIR / "missing_values_report.csv", index=False
)

# Save duplicate records separately before removing exact duplicates.
df.loc[df.duplicated(keep=False)].to_csv(
    REPORT_DIR / "duplicate_records_for_review.csv",
    index=False
)

print("\nExact duplicate rows:", exact_duplicate_count)

# Remove only exact duplicate rows.
# Do not remove rows merely because an ID repeats:
# a single order can contain multiple order items.
df = df.drop_duplicates().copy()

# ---------- 4. Validate business fields ----------
# Recognize common DataCo column names after standardization.
numeric_candidates = [
    "sales",
    "profit",
    "order_profit_per_order",
    "order_item_quantity",
    "order_item_product_price",
    "order_item_discount",
    "order_item_discount_rate",
    "order_item_total",
    "days_for_shipping_real",
    "days_for_shipment_scheduled",
    "late_delivery_risk",
    "benefit_per_order",
    "sales_per_customer"
]

numeric_report = []

for col in numeric_candidates:
    if col not in df.columns:
        continue

    converted = pd.to_numeric(df[col], errors="coerce")
    failed = int((df[col].notna() & converted.isna()).sum())

    df[col] = converted

    numeric_report.append({
        "column": col,
        "failed_numeric_conversions": failed,
        "negative_values": int((df[col] < 0).sum()),
        "zero_values": int((df[col] == 0).sum()),
        "missing_after_conversion": int(df[col].isna().sum())
    })

pd.DataFrame(numeric_report).to_csv(
    REPORT_DIR / "numeric_validation_report.csv",
    index=False,
 )

# Flag suspicious records; do not delete them automatically.
flags = pd.DataFrame(index=df.index)

if "sales" in df.columns:
    flags["negative_sales"] = df["sales"] < 0

if "order_item_quantity" in df.columns:
    flags["nonpositive_quantity"] = (
        df["order_item_quantity"] <= 0
    )

if "order_item_discount_rate" in df.columns:
    flags["discount_rate_outside_0_1"] = (
        ~df["order_item_discount_rate"].between(0, 1)
        & df["order_item_discount_rate"].notna()
    )

if "late_delivery_risk" in df.columns:
    flags["invalid_delivery_risk"] = (
        ~df["late_delivery_risk"].isin([0, 1])
        & df["late_delivery_risk"].notna()
    )

if {
    "days_for_shipping_real",
    "days_for_shipment_scheduled"
}.issubset(df.columns):
    flags["negative_shipping_days"] = (
        (df["days_for_shipping_real"] < 0)
        | (df["days_for_shipment_scheduled"] < 0)
    )

if not flags.empty:
    flags = flags.fillna(False).astype(bool)
    df["quality_review_needed"] = flags.any(axis=1)

    review = df.loc[df["quality_review_needed"]].copy()
    review.to_csv(
        REPORT_DIR / "records_for_review.csv", index=False
    )

    print("\nRecords flagged for review:", len(review))

# ---------- 5. Save cleaned dataset ----------
df.to_csv(OUTPUT_FILE, index=False)

print("\nCleaning completed.")
print("Final dataset shape:", df.shape)
print("Cleaned file:", OUTPUT_FILE)
print("Reports folder:", REPORT_DIR)


FileNotFoundError: Dataset not found: c:\Project\Data Analyst\New folder\supply-chain-analytics\data\your_dataset.csv
Update INPUT_FILE with your actual CSV filename.

In [3]:
# ---------- 1. Standardize column names ----------
def standardize_column(name):
    name = str(name).strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")

df.columns = [standardize_column(c) for c in df.columns]

NameError: name 're' is not defined